In [ ]:
import geopandas as gpd
import pandas as pd

1. ## Load Data

In [ ]:
#import both datasets with AHP_SCORE AND SPP_PRESENCE
socio_env = gpd.read_file ("/Users/eliviau/Documents/hex2vec/SSPpresenceandscoresocioenvhokkaido3.geojson")
techno_econ = gpd.read_file ("/Users/eliviau/Documents/hex2vec/SSPpresenceandscorestechnohokkaidoFINAL.geojson")

In [ ]:
socio_env.head()

In [ ]:
techno_econ.head()

In [ ]:

# Rename AHP total scores
socio_env = socio_env.rename(columns={'ahp_total_score': 'ahp_total_score_env'})
techno_econ = techno_econ.rename(columns={'ahp_total_score': 'ahp_total_score_techno'})


In [ ]:
#create a merged dataframe with the 2 datasets
merged = pd.merge(
    socio_env[['region_id', 'ahp_total_score_env', 'SPP_presence', 'geometry']],
    techno_econ[['region_id', 'ahp_total_score_techno']],
    on='region_id'
)


In [ ]:
merged.head()

# 2. Build spatial weight 
Using QUEEN contiguity

In [ ]:
# Project to a metric CRS (needed for spatial weights)
merged_proj = merged.to_crs(epsg=3857)


In [ ]:
from libpysal.weights import Queen
from esda.moran import Moran_BV
import numpy as np

# Build weights
w = Queen.from_dataframe(merged_proj)
w.transform = 'r'  # Row-standardize the weights


## CALCULATE BIVARIATE MORAN'S I 

In [ ]:
from libpysal.weights import Queen
from esda.moran import Moran_BV
import numpy as np

# Step 1: Drop NA and reset index
df_env = merged_proj.dropna(subset=['ahp_total_score_env', 'SPP_presence']).reset_index(drop=True)

# Step 2: Rebuild Queen weights based on the subset only
w_env = Queen.from_dataframe(df_env)
w_env.transform = 'r'

# Step 3: Extract variables
x_env = df_env['ahp_total_score_env'].values
y_spp = df_env['SPP_presence'].values

# Step 4: Run Bivariate Moran
moran_env = Moran_BV(x_env, y_spp, w_env)

# Step 5: Output
print("Env vs SPP")
print("Moran's I:", moran_env.I)
print("p-value:", moran_env.p_sim)


In [ ]:
# Step 1: Drop NA and reset index
df_tech = merged_proj.dropna(subset=['ahp_total_score_techno', 'SPP_presence']).reset_index(drop=True)

# Step 2: Rebuild weights on this subset
w_tech = Queen.from_dataframe(df_tech)
w_tech.transform = 'r'

# Step 3: Extract variables
x_tech = df_tech['ahp_total_score_techno'].values
y_spp = df_tech['SPP_presence'].values

# Step 4: Run Bivariate Moran
moran_tech = Moran_BV(x_tech, y_spp, w_tech)

# Step 5: Output
print("Techno vs SPP")
print("Moran's I:", moran_tech.I)
print("p-value:", moran_tech.p_sim)


## ADVANCED ANALYSIS

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from splot.esda import moran_scatterplot


# Techno suitability
fig1, ax1 = moran_scatterplot(moran_tech, aspect_equal=True)
ax1.set_title("Moran Scatterplot: Techno Suitability vs. SPP")
plt.show()

# Environmental suitability
fig2, ax2 = moran_scatterplot(moran_env, aspect_equal=True)
ax2.set_title("Moran Scatterplot: Env Suitability vs. SPP")
plt.show()


In [ ]:
from esda.moran import Moran_Local_BV
import matplotlib.pyplot as plt
import geopandas as gpd

# Reuse techno data and weights
x = df_tech['ahp_total_score_techno'].values
y = df_tech['SPP_presence'].values

# Compute local Moran's I (bivariate)
lisa = Moran_Local_BV(x, y, w_tech)

# Add LISA values to the GeoDataFrame
df_tech['lisa_I'] = lisa.Is
df_tech['lisa_p'] = lisa.p_sim
df_tech['lisa_q'] = lisa.q

# Define cluster labels
labels = {
    1: "High Techno - High SPP (HH)",
    2: "Low Techno - High SPP (LH)",
    3: "Low Techno - Low SPP (LL)",
    4: "High Techno - Low SPP (HL)"
}
df_tech['lisa_cluster'] = df_tech['lisa_q'].map(labels)


In [ ]:
from matplotlib.colors import ListedColormap  # <-- Add this at the top

# Color map
color_dict = {
    "High Techno - High SPP (HH)": "green",
    "Low Techno - High SPP (LH)": "red",
    "Low Techno - Low SPP (LL)": "grey",
    "High Techno - Low SPP (HL)": "lightblue"
}

# Plot
fig, ax = plt.subplots(1, 1, figsize=(10, 8))
df_tech.plot(
    column='lisa_cluster',
    categorical=True,
    legend=True,
    cmap=ListedColormap([color_dict[k] for k in labels.values()]),
    ax=ax,
    edgecolor='none',
    linewidth=0.3
)
ax.set_title("LISA Cluster Map: Techno Suitability vs. SPP (Significant only)")
plt.axis("off")
plt.show()


In [ ]:
spp_location = gpd.read_file("/Users/eliviau/Documents/hex2vec/power_plants_hokkaido.geojson")

In [ ]:
import folium
import geopandas as gpd
from esda.moran import Moran_Local_BV
import matplotlib.pyplot as plt
import geopandas as gpd
from matplotlib.colors import ListedColormap  # <-- Add this at the top
from branca.element import Template, MacroElement

# 1. Reproject to WGS84 (required for Folium)
df_tech = df_tech.to_crs(epsg=4326)

# 1. Define your color dictionary
color_dict = {
    "High Techno - High SPP (HH)": "green",
    "Low Techno - High SPP (LH)": "red",
    "Low Techno - Low SPP (LL)": "grey",
    "High Techno - Low SPP (HL)": "lightblue"
}

# 2. Initialize Folium map centered on your data
center = df_tech.geometry.unary_union.centroid
m = folium.Map(location=[center.y, center.x], zoom_start=7, tiles='cartodbpositron')

# Replace this loop:
for _, row in df_tech.iterrows():
    cluster = row['lisa_cluster']
    hex_id = row['region_id']  # 🔁 Replace with your actual hex ID column name
    color = color_dict.get(cluster, 'gray')
    sim_geo = gpd.GeoSeries(row['geometry']).simplify(0.001)
    geo_j = sim_geo.to_json()

    tooltip_text = f"Hex ID: {hex_id}<br>Cluster: {cluster}"

    folium.GeoJson(
        data=geo_j,
        style_function=lambda x, color=color: {
            'fillColor': color,
            'color': color,
            'weight': 0.5,
            'fillOpacity': 0.5
        },
        tooltip=folium.Tooltip(tooltip_text, sticky=True)

    ).add_to(m)

# (5) Add ALL power plants in black
spp_location.explore(m=m, color='black', marker_kwds={'radius': 4, 'fill': True})

legend_html = """
{% macro html(this, kwargs) %}
<div style='
    position: fixed;
    bottom: 50px;
    left: 50px;
    width: 220px;
    z-index:9999;
    font-size:14px;
    background-color: white;
    padding: 10px;
    border: 2px solid gray;
    border-radius: 5px;
    box-shadow: 2px 2px 6px rgba(0,0,0,0.3);
'>
    <strong>LISA Cluster Legend</strong><br>
    <i style='background: red; width:10px; height:10px; float:left; margin-right:5px;'></i> HH<br>
    <i style='background: lightblue; width:10px; height:10px; float:left; margin-right:5px;'></i> HL<br>
    <i style='background: orange; width:10px; height:10px; float:left; margin-right:5px;'></i> LH<br>
    <i style='background: blue; width:10px; height:10px; float:left; margin-right:5px;'></i> LL<br>
</div>
{% endmacro %}
"""

legend = MacroElement()
legend._template = Template(legend_html)
m.get_root().add_child(legend)

# 5. Show or save
m.save("lisa_interactive_map.html")
m


In [ ]:
from esda.moran import Moran_Local_BV
import matplotlib.pyplot as plt
import geopandas as gpd

# Reuse techno data and weights
x = df_env['ahp_total_score_env'].values
y = df_env['SPP_presence'].values

# Compute local Moran's I (bivariate)
lisa = Moran_Local_BV(x, y, w_env)

# Add LISA values to the GeoDataFrame
df_env['lisa_I'] = lisa.Is
df_env['lisa_p'] = lisa.p_sim
df_env['lisa_q'] = lisa.q

# Define cluster labels
labels = {
    1: "High env - High SPP (HH)",
    2: "Low env - High SPP (LH)",
    3: "Low env - Low SPP (LL)",
    4: "High env - Low SPP (HL)"
}
df_env['lisa_cluster'] = df_env['lisa_q'].map(labels)


In [ ]:
from matplotlib.colors import ListedColormap  # <-- Add this at the top

# Color map
color_dict = {
    "High env - High SPP (HH)": "red",
    "Low env - High SPP (LH)": "lightblue",
    "Low env - Low SPP (LL)": "blue",
    "High env - Low SPP (HL)": "orange"
}

# Plot
fig, ax = plt.subplots(1, 1, figsize=(10, 8))
df_env.plot(
    column='lisa_cluster',
    categorical=True,
    legend=True,
    cmap=ListedColormap([color_dict[k] for k in labels.values()]),
    ax=ax,
    edgecolor='none',
    linewidth=0.3
)
ax.set_title("LISA Cluster Map: ENV Suitability vs. SPP (Significant only)")
plt.axis("off")
plt.show()
